# Lab 3 — DBSCAN Clusters and Noise

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

Use density clustering to detect core clusters and noisy outlier symbols.

<!-- cisco-day05-lab03-expanded-2026 -->

**Dataset:** `data/nyse/nyse_stocks.csv` (500 rows, 25 symbols)

## Syllabus note — Spectral & OPTICS

The course also mentions **Spectral Clustering** and **OPTICS**. This notebook keeps the original comparison section.

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])
print(f"symbols: {len(features)}")


## Fit DBSCAN

In [ ]:
EPS = 1.2
MIN_SAMPLES = 3

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

unique, counts = np.unique(labels, return_counts=True)
label_counts = dict(zip(unique.tolist(), counts.tolist()))

print("Lab 3 — DBSCAN clusters")
print(f"eps: {EPS}, min_samples: {MIN_SAMPLES}")
print(f"clusters found: {n_clusters}")
print(f"noise points: {n_noise}")
print(f"label counts: {label_counts}")


## 2b. Compare Spectral & OPTICS (course topic)

In [ ]:
from sklearn.cluster import OPTICS, SpectralClustering


def cluster_summary(name: str, lbl: np.ndarray) -> None:
    noise = int(np.sum(lbl == -1))
    clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    print(f"{name:18s} clusters={clusters}, noise={noise}")


spec_labels = SpectralClustering(
    n_clusters=3, affinity="nearest_neighbors", n_neighbors=5, random_state=42
).fit_predict(X_scaled)
optics_labels = OPTICS(min_samples=MIN_SAMPLES).fit_predict(X_scaled)

cluster_summary("DBSCAN", labels)
cluster_summary("Spectral (k=3)", spec_labels)
cluster_summary("OPTICS", optics_labels)
print("Spectral: graph-based — good for non-convex groups.")
print("OPTICS: ordering by density — reveals varying-density hierarchies.")


## Attach labels to feature table

In [ ]:
features = features.copy()
features["dbscan_label"] = labels

display(
    features[["symbol", "avg_close", "volatility", "dbscan_label"]]
    .sort_values("dbscan_label")
    .round(2)
)


## Visualize DBSCAN labels

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features["avg_close"],
    features["volatility"],
    c=features["dbscan_label"],
    cmap="tab10",
    s=100,
    alpha=0.85,
)
for _, row in features.iterrows():
    color = "gray" if row["dbscan_label"] == -1 else "black"
    ax.annotate(row["symbol"], (row["avg_close"], row["volatility"]), fontsize=8, color=color, alpha=0.8)
ax.set_xlabel("avg_close")
ax.set_ylabel("volatility")
ax.set_title(f"DBSCAN (eps={EPS}, noise={n_noise})")
fig.colorbar(scatter, ax=ax, label="cluster (-1 = noise)")
plt.tight_layout()
plt.show()


## Compare with K-Means cluster counts

In [ ]:
kmeans_labels = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)
kmeans_counts = dict(zip(*np.unique(kmeans_labels, return_counts=True)))

compare = pd.DataFrame({
    "method": ["K-Means (k=4)", "DBSCAN"],
    "clusters": [4, n_clusters],
    "noise_or_unassigned": [0, n_noise],
    "label_counts": [str(kmeans_counts), str(label_counts)],
})
display(compare)


## Sensitivity study for epsilon

In [ ]:
rows = []
for eps in [0.8, 1.2, 1.5]:
    lbl = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X_scaled)
    n_clust = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_no = int(np.sum(lbl == -1))
    rows.append({"eps": eps, "clusters": n_clust, "noise": n_no})

display(pd.DataFrame(rows))


### Density clustering prompt 1

Show the noise symbols list.

In [ ]:
display(features.loc[features['dbscan_label']==-1, ['symbol','avg_close','volatility']].round(2))

### Density clustering prompt 2

Count non-noise symbols.

In [ ]:
print(int((features['dbscan_label']!=-1).sum()))

### Density clustering prompt 3

Compute mean volatility among noise points.

In [ ]:
print(round(features.loc[features['dbscan_label']==-1, 'volatility'].mean(), 3))

### Density clustering prompt 4

Compute mean avg_close among clustered points.

In [ ]:
print(round(features.loc[features['dbscan_label']!=-1, 'avg_close'].mean(), 2))

### Density clustering prompt 5

Sort label counts by key.

In [ ]:
print(dict(sorted(label_counts.items(), key=lambda x: x[0])))

### Density clustering prompt 6

Cross-check DBSCAN clusters count formula.

In [ ]:
print(len(set(labels)) - (1 if -1 in labels else 0))

### Density clustering prompt 7

Try a stricter min_samples quickly.

In [ ]:
labels_ms4 = DBSCAN(eps=EPS, min_samples=4).fit_predict(X_scaled); print(dict(zip(*np.unique(labels_ms4, return_counts=True))))

### Density clustering prompt 8

Inspect symbols from cluster 0.

In [ ]:
display(features.loc[features['dbscan_label']==0, ['symbol','avg_close','volatility']].round(2))

### Density clustering prompt 9

Inspect symbols from cluster 1.

In [ ]:
display(features.loc[features['dbscan_label']==1, ['symbol','avg_close','volatility']].round(2))

### Density clustering prompt 10

Inspect symbols from cluster 2.

In [ ]:
display(features.loc[features['dbscan_label']==2, ['symbol','avg_close','volatility']].round(2))

### Density clustering prompt 11

Compare kmeans vs dbscan assignment for first 10 symbols.

In [ ]:
cmp = features[['symbol','dbscan_label']].copy(); cmp['kmeans_label']=kmeans_labels; display(cmp.head(10))

### Density clustering prompt 12

Compute agreement where DBSCAN is not noise.

In [ ]:
tmp = features.copy(); tmp['kmeans']=kmeans_labels; mask = tmp['dbscan_label']!=-1; print(round((tmp.loc[mask,'dbscan_label']==tmp.loc[mask,'kmeans']).mean(), 3))

### Density clustering prompt 13

Show spread stats by dbscan label.

In [ ]:
display(features.groupby('dbscan_label')[FEATURE_COLUMNS].agg(['mean','std']).round(2))

### Density clustering prompt 14

Check cluster_summary helper again.

In [ ]:
cluster_summary('DBSCAN repeat', labels)

### Density clustering prompt 15

Count labels with pandas.

In [ ]:
print(features['dbscan_label'].value_counts().sort_index().to_dict())

### Density clustering prompt 16

Display label histogram.

In [ ]:
ax = features['dbscan_label'].plot(kind='hist', bins=7, figsize=(5,3)); ax.set_title('DBSCAN label histogram');

### Density clustering prompt 17

Summarize current parameters.

In [ ]:
print({'EPS': EPS, 'MIN_SAMPLES': MIN_SAMPLES, 'n_clusters': n_clusters, 'n_noise': n_noise})

### Density clustering prompt 18

Bridge to cluster metrics lab.

In [ ]:
print('Next: evaluate cluster quality with silhouette, DB, and CH scores.')

### Density clustering prompt 19

Verify expected symbol count.

In [ ]:
print(len(features), features['symbol'].nunique())

### Density clustering prompt 20

Review top 5 highest-volatility symbols.

In [ ]:
display(features.nlargest(5, 'volatility')[['symbol','volatility','dbscan_label']].round(3))

### Lab 3 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 3 recap step 1: completed")

### Lab 3 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 3 recap step 2: completed")

### Lab 3 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 3 recap step 3: completed")

## Final checkpoint

In [ ]:
assert len(features) == 25
assert n_clusters == 3
assert n_noise == 11
assert label_counts[-1] == 11
assert label_counts[0] == 6 and label_counts[1] == 4 and label_counts[2] == 4
print("✓ All checkpoint assertions passed")


## Reflection

Why is explicit noise labeling useful for market anomaly watchlists?